# VisualLand-AI - Step-by-step execution

## 1. Imports

In [ ]:
import hashlib
from pathlib import Path
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split


## 2. Dataset setup and exploration

In [ ]:
dataset_root = Path("data/Intel_Image_Classification")
train_dir = dataset_root / "seg_train_test" / "seg_train"
if not train_dir.exists():
    raise FileNotFoundError(f"Train directory not found: {train_dir}")
for path in train_dir.iterdir():
    if path.is_dir():
        print("Directory:", path.name)

image_classes = {"buildings", "forest", "glacier", "mountain", "sea", "street"}
class_count = {}
image_distribution = {}
total_images = 0
for class_folder in train_dir.iterdir():
    if class_folder.is_dir():
        class_count[class_folder.name] = sum(1 for image_path in class_folder.rglob("*") if image_path.is_file())
for class_name in image_classes:
    total_images += class_count.get(class_name)
for class_name in sorted(image_classes):
    image_count = class_count.get(class_name, 0)
    image_distribution[class_name] = (image_count / total_images) * 100 if total_images > 0 else 0
print("Total images:", total_images)
print(f"Image distribution: {image_distribution[class_name]:.2f}")

min_width = None
min_height = None
images = []
for class_folder in train_dir.iterdir():
    if class_folder.is_dir():
        for image_path in class_folder.glob("*"):
            if image_path.is_file() and image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                with Image.open(image_path) as image:
                    width, height = image.size
                if min_width is None or width < min_width:
                    min_width = width
                if min_height is None or height < min_height:
                    min_height = height
                images.append(image_path)
            else:
                print("Images not found!")
hashes = {}
duplicates = []
unique_images = []
for image_path in images:
    file_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
    if file_hash in hashes:
        duplicates.append((hashes[file_hash], image_path))
    else:
        hashes[file_hash] = image_path
        unique_images.append(image_path)

print(min_width, min_height)
print(class_count)
print(image_distribution)
plt.figure(figsize=(16, 16))
plt.bar(image_distribution.keys(), image_distribution.values())
plt.title("Image distribution per class")
plt.xlabel("Image classes")
plt.ylabel("Percentage of images")
plt.show()
plt.figure(figsize=(16, 16))
plt.bar(class_count.keys(), class_count.values())
plt.title("Image count per class")
plt.xlabel("Number of images per class")
plt.ylabel("Classes")
plt.show()



## 3. Duplicate detection and dataset split

In [ ]:
print("Duplicated images display!")
print("=" * 50)
for original, duplicated in duplicates:
    print("\nOriginal:", original, "and its duplicate:", duplicated)
print("=" * 50)
print("Non-duplicated images display!")
print("=" * 50)
for image_path in unique_images:
    print("\nUnique image example:", image_path)
print("=" * 50)
print("Split image data into train, validation and test!")
labels = [image_path.parent.name for image_path in unique_images]
train, temporary_values = train_test_split(unique_images, test_size=0.2, random_state=42, stratify=labels)
temporary_labels = [image_path.parent.name for image_path in temporary_values]
validation, test = train_test_split(temporary_values, test_size=0.5, random_state=42, stratify=temporary_labels)
print("=" * 50)


## 4. Image resizing

In [ ]:
print("Data preprocessing!")
print("=" * 50)

trained_data = []
for image_path in train:
    image = tf.keras.utils.load_img(str(image_path))
    trained_data.append(tf.keras.preprocessing.image.smart_resize(tf.keras.utils.img_to_array(image), (256, 256)))
validation_data = []
for image_path in validation:
    image = tf.keras.utils.load_img(str(image_path))
    validation_data.append(tf.keras.preprocessing.image.smart_resize(tf.keras.utils.img_to_array(image), (256, 256)))
test_data = []
for image_path in test:
    image = tf.keras.utils.load_img(str(image_path))
    test_data.append(tf.keras.preprocessing.image.smart_resize(tf.keras.utils.img_to_array(image), (256, 256)))
print("Trained data resized: ", trained_data)
print("\nValidation data resized: ", validation_data)
print("\nTest data resized: ", test_data)



## 5. Normalization

In [ ]:
normalized_train_data = [tf.cast(image, tf.float32) / 255.0 for image in trained_data]
normalized_validation_data = [tf.cast(image, tf.float32) / 255.0 for image in validation_data]
normalized_test_data = [tf.cast(image, tf.float32) / 255.0 for image in test_data]
print("Trained data normalized: ", normalized_train_data)
print("\nValidation data normalized: ", normalized_validation_data)
print("\nTest data normalized: ", normalized_test_data)

print("=" * 50)


## 6. Batch loading

In [ ]:
print("Batch loading!")
class_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
trained_labels = [class_names.index(image_path.parent.name) for image_path in train]
validation_labels = [class_names.index(image_path.parent.name) for image_path in validation]
test_labels = [class_names.index(image_path.parent.name) for image_path in test]
batched_trained_dataset = tf.data.Dataset.from_generator(lambda: zip(normalized_train_data, trained_labels), output_signature=(tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.int32))).shard(num_shards=5, index=0).shuffle(buffer_size=1000).batch(32).prefetch(tf.data.AUTOTUNE)
batched_validation_dataset = tf.data.Dataset.from_generator(lambda: zip(normalized_validation_data, validation_labels), output_signature=(tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.int32))).shard(num_shards=5, index=0).shuffle(buffer_size=1000).batch(32).prefetch(tf.data.AUTOTUNE)
batched_test_dataset = tf.data.Dataset.from_generator(lambda: zip(normalized_test_data, test_labels), output_signature=(tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.int32))).shard(num_shards=5, index=0).shuffle(buffer_size=1000).batch(32).prefetch(tf.data.AUTOTUNE)
print("Batched trained dataset: ", batched_trained_dataset)
print("\nBatched validation dataset: ", batched_validation_dataset)
print("\nBatched test dataset: ", batched_test_dataset)
print("=" * 50)



## 7. Data augmentation and visualization

In [ ]:
print("Data augmentation!")
data_augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal_and_vertical"), tf.keras.layers.RandomRotation(0.2), tf.keras.layers.RandomZoom(0.2)])
augmented_trained_dataset = batched_trained_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))
print("Augmented trained dataset: ", augmented_trained_dataset)
for image, label in augmented_trained_dataset.take(1):
    plt.figure(figsize=(16, 16))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(image[i].numpy())
        plt.title(int(label[i]))
        plt.axis("off")
    plt.show()
print("=" * 50)


## 8. Model building and training

In [ ]:
print("Model building!")
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(6, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
print("=" * 50)
print("Model training!")
history = model.fit(augmented_trained_dataset, epochs=10)
print("Model training history: ", history)
